# Yambda multi-event: EDA, аномалии и подготовка clean-слоя

Ноутбук анализирует `multi_event.parquet` со схемой `uid`, `item_id`, `timestamp`, `is_organic`, `event_type`, `played_ratio_pct`, `track_length_seconds`. Все тяжёлые операции построены на ленивом API Polars и streaming engine.

Этапы: загрузка и проверка схемы → базовый EDA → пользовательские, контентные и временные срезы → визуализация → общие и доменные аномалии → clean-слой → нормализованные state-переходы и snapshots.

In [ ]:
from pathlib import Path
import math
import shutil

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display
from matplotlib.ticker import FuncFormatter, PercentFormatter

PROJECT_DIR = Path.cwd()
LOCAL_SOURCE_PATH = PROJECT_DIR / "data" / "raw" / "multi_event.parquet"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURES_DIR = PROJECT_DIR / "reports" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_COLUMNS = [
    "uid", "timestamp", "item_id", "is_organic",
    "played_ratio_pct", "track_length_seconds", "event_type",
]
EVENT_TYPES = ["listen", "like", "unlike", "dislike", "undislike"]
HARD_MAX_TRACK_SECONDS = 4 * 60 * 60

if not LOCAL_SOURCE_PATH.is_file():
    from huggingface_hub import hf_hub_download
    cached = hf_hub_download(
        repo_id="yandex/yambda",
        filename="flat/500m/multi_event.parquet",
        repo_type="dataset",
    )
    LOCAL_SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cached, LOCAL_SOURCE_PATH)

SOURCE_PATH = LOCAL_SOURCE_PATH
print(f"Polars {pl.__version__}; source={SOURCE_PATH}; size={SOURCE_PATH.stat().st_size / 1024**2:.1f} MiB")

## 1. Как работать с этим датасетом в Polars

`scan_parquet` возвращает `LazyFrame`: чтение откладывается до `collect` или `sink_parquet`. Выражения `select`, `with_columns`, `filter`, `group_by().agg()` и `sort` формируют оптимизируемый план. Для 47+ млн строк используется `collect(engine="streaming")`, чтобы обрабатывать данные пакетами.

Важно: `played_ratio_pct` и `track_length_seconds` заполнены только для `listen`. Их `null` у `like`, `unlike`, `dislike`, `undislike` является корректной структурой, а не ошибкой качества.

In [ ]:
events_lf = pl.scan_parquet(SOURCE_PATH)
schema = events_lf.collect_schema()
missing_columns = set(EXPECTED_COLUMNS) - set(schema.names())
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

overview = events_lf.select(
    pl.len().alias("rows"),
    pl.col("uid").n_unique().alias("users"),
    pl.col("item_id").n_unique().alias("items"),
    pl.col("timestamp").min().alias("timestamp_min"),
    pl.col("timestamp").max().alias("timestamp_max"),
    pl.col("event_type").n_unique().alias("event_types"),
).collect(engine="streaming")
null_profile = events_lf.select(pl.all().null_count()).collect(engine="streaming")

print(schema)
display(overview)
display(null_profile)

In [ ]:
# Вычисляемые поля остаются частью ленивого плана.
analysis_lf = events_lf.with_columns(
    (pl.col("timestamp").cast(pl.UInt64) // (24*60*60))
    .cast(pl.UInt32)
    .alias("time_period"),
    (pl.col("event_type") == "listen").alias("is_listen"),
    pl.when(pl.col("event_type") == "listen")
    .then(
        pl.col("track_length_seconds").cast(pl.Float64)
        * pl.col("played_ratio_pct").cast(pl.Float64) / 100
    )
    .otherwise(None)
    .alias("played_seconds_raw"),
    pl.when(pl.col("event_type") == "listen")
    .then(
        pl.col("track_length_seconds").cast(pl.Float64)
        * pl.col("played_ratio_pct").clip(0, 100).cast(pl.Float64) / 100
    )
    .otherwise(None)
    .alias("played_seconds_capped"),
)

print(analysis_lf.select("uid", "event_type", "time_period", "played_seconds_raw").limit(5).collect(engine="streaming"))

## 2. Базовый EDA

Сначала рассматриваются объёмы событий, пользователи и треки, затем playback-метрики только для `listen`. Для действий playback-поля не агрегируются как нули: структурные `null` сохраняют правильную семантику.

In [ ]:
event_summary = (
    events_lf.group_by("event_type")
    .agg(
        pl.len().alias("events"),
        pl.col("uid").n_unique().alias("users"),
        pl.col("item_id").n_unique().alias("items"),
        pl.col("is_organic").mean().alias("organic_share"),
        pl.col("played_ratio_pct").mean().alias("avg_played_ratio_pct"),
    )
    .sort("events", descending=True)
    .collect(engine="streaming")
)
listen_profile = (
    events_lf.filter(pl.col("event_type") == "listen")
    .select(
        pl.col("played_ratio_pct").min().alias("ratio_min"),
        pl.col("played_ratio_pct").median().alias("ratio_median"),
        pl.col("played_ratio_pct").mean().alias("ratio_mean"),
        pl.col("played_ratio_pct").quantile(0.99, interpolation="nearest").alias("ratio_p99"),
        pl.col("played_ratio_pct").max().alias("ratio_max"),
        pl.col("track_length_seconds").min().alias("length_min"),
        pl.col("track_length_seconds").median().alias("length_median"),
        pl.col("track_length_seconds").mean().alias("length_mean"),
        pl.col("track_length_seconds").quantile(0.99, interpolation="nearest").alias("length_p99"),
        pl.col("track_length_seconds").max().alias("length_max"),
    )
    .collect(engine="streaming")
)
display(event_summary)
display(listen_profile)

In [ ]:
daily_lf = analysis_lf.group_by("time_period").agg(
    pl.len().alias("events"),
    pl.col("uid").n_unique().alias("dau"),
    pl.col("is_organic").mean().alias("organic_share"),
    pl.col("is_listen").sum().alias("listens"),
    (pl.col("event_type") == "like").sum().alias("likes"),
    pl.col("played_seconds_capped").sum().alias("played_seconds"),
).sort("time_period")
daily = daily_lf.collect(engine="streaming")

user_lf = analysis_lf.group_by("uid").agg(
    pl.len().alias("events"),
    pl.col("item_id").n_unique().alias("unique_items"),
    pl.col("is_listen").sum().alias("listens"),
    (pl.col("event_type") == "like").sum().alias("likes"),
    (pl.col("event_type") == "dislike").sum().alias("dislikes"),
    (pl.col("played_ratio_pct") >= 90).sum().alias("completed_listens"),
    (pl.col("played_seconds_capped").sum() / 3600).alias("played_hours"),
)
item_lf = analysis_lf.group_by("item_id").agg(
    pl.len().alias("events"),
    pl.col("uid").n_unique().alias("users"),
    pl.col("is_listen").sum().alias("listens"),
    (pl.col("event_type") == "like").sum().alias("likes"),
    (pl.col("event_type") == "dislike").sum().alias("dislikes"),
    pl.col("played_ratio_pct").mean().alias("avg_played_ratio_pct"),
    pl.col("track_length_seconds").max().alias("track_length_seconds"),
)

top_users = user_lf.sort("events", descending=True).limit(10).collect(engine="streaming")
top_items = item_lf.sort("listens", descending=True).limit(10).collect(engine="streaming")
display(daily.select(pl.all().exclude("played_seconds")).head(10))
display(top_users)
display(top_items)

In [ ]:
listen_kpis = (
    analysis_lf.filter(pl.col("event_type") == "listen")
    .select(
        pl.len().alias("listens"),
        (pl.col("played_ratio_pct") < 10).sum().alias("short_listens_lt_10pct"),
        (pl.col("played_ratio_pct") >= 90).sum().alias("completed_listens_ge_90pct"),
        (pl.col("played_ratio_pct") > 100).sum().alias("ratio_over_100pct"),
        (pl.col("played_seconds_raw").sum() / 3600).alias("raw_played_hours"),
        (pl.col("played_seconds_capped").sum() / 3600).alias("capped_played_hours"),
    )
    .collect(engine="streaming")
)
display(listen_kpis)

## 3. Визуализация EDA

Для тяжёлых распределений сначала выполняется агрегация Polars, поэтому Matplotlib получает только компактные таблицы частот, а не десятки миллионов строк. Графики одновременно сохраняются в `reports/figures/`.

In [ ]:
plt.style.use("dark_background")
ratio_distribution = (
    events_lf.filter(pl.col("event_type") == "listen")
    .group_by("played_ratio_pct").agg(pl.len().alias("events"))
    .sort("played_ratio_pct").collect(engine="streaming")
)
length_distribution = (
    events_lf.filter(pl.col("event_type") == "listen")
    .group_by("track_length_seconds").agg(pl.len().alias("events"))
    .sort("track_length_seconds").collect(engine="streaming")
)
daily_plot = daily.with_columns(
    pl.col("events").rolling_mean(window_size=30, min_samples=1).alias("events_30d"),
    pl.col("dau").rolling_mean(window_size=30, min_samples=1).alias("dau_30d"),
)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Yambda multi-event: обзор датасета", fontsize=20, fontweight="bold")

event_names = event_summary["event_type"].cast(pl.String).to_list()
event_counts = event_summary["events"].to_numpy()
axes[0, 0].bar(event_names, event_counts, color=["#55C2D9", "#FFCC00", "#F28E2B", "#B07AA1", "#59A14F"])
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("События по типам (log)")
axes[0, 0].tick_params(axis="x", rotation=20)
axes[0, 0].grid(axis="y", alpha=0.25)

axes[0, 1].plot(daily_plot["time_period"], daily_plot["events_30d"], color="#FFCC00", label="events, MA30")
right_axis = axes[0, 1].twinx()
right_axis.plot(daily_plot["time_period"], daily_plot["dau_30d"], color="#55C2D9", label="DAU, MA30")
axes[0, 1].set_title("Динамика событий и DAU")
axes[0, 1].set_xlabel("условный день")
axes[0, 1].set_ylabel("события")
right_axis.set_ylabel("DAU")
axes[0, 1].grid(alpha=0.25)

ratio_x = ratio_distribution["played_ratio_pct"].to_numpy()
ratio_y = ratio_distribution["events"].to_numpy()
axes[1, 0].bar(ratio_x, ratio_y, width=1.0, color=np.where(ratio_x > 100, "#F28E2B", "#59A14F"))
axes[1, 0].axvline(100, color="#FFCC00", linestyle="--", label="100%")
axes[1, 0].set_title("Распределение played_ratio_pct")
axes[1, 0].set_xlabel("процент прослушивания")
axes[1, 0].set_ylabel("события")
axes[1, 0].legend()
axes[1, 0].grid(axis="y", alpha=0.25)

length_p99 = int(listen_profile[0, "length_p99"])
visible_lengths = length_distribution.filter(pl.col("track_length_seconds") <= length_p99)
axes[1, 1].plot(visible_lengths["track_length_seconds"], visible_lengths["events"], color="#B07AA1")
axes[1, 1].axvline(length_p99, color="#FFCC00", linestyle="--", label=f"p99={length_p99}s")
axes[1, 1].set_title("Длительность треков до p99")
axes[1, 1].set_xlabel("секунды")
axes[1, 1].set_ylabel("прослушивания")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.25)

fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(FIGURES_DIR / "multi-event-eda.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
completion_bands = (
    events_lf.filter(pl.col("event_type") == "listen")
    .with_columns(
        pl.when(pl.col("played_ratio_pct") < 10).then(pl.lit("0-9%"))
        .when(pl.col("played_ratio_pct") < 50).then(pl.lit("10-49%"))
        .when(pl.col("played_ratio_pct") < 90).then(pl.lit("50-89%"))
        .when(pl.col("played_ratio_pct") <= 100).then(pl.lit("90-100%"))
        .otherwise(pl.lit(">100%")).alias("band")
    )
    .group_by("band").agg(pl.len().alias("events"))
    .collect(engine="streaming")
)
band_order = ["0-9%", "10-49%", "50-89%", "90-100%", ">100%"]
band_lookup = dict(zip(completion_bands["band"], completion_bands["events"]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(event_names, event_summary["organic_share"], color="#55C2D9")
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].set_title("Органическая доля по типу события")
axes[0].tick_params(axis="x", rotation=20)
axes[0].grid(axis="y", alpha=0.25)

band_values = [band_lookup.get(name, 0) for name in band_order]
axes[1].bar(band_order, band_values, color=["#F28E2B", "#B07AA1", "#55C2D9", "#59A14F", "#FFCC00"])
axes[1].set_title("Глубина прослушивания")
axes[1].set_ylabel("события listen")
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "multi-event-behavior.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Поиск аномалий с учётом домена

Проверки разделены по смыслу:

- **invalid** — нарушение схемы или физического ограничения: пропущенный core-field, неправильный `is_organic`, playback-поля не у того типа события, отрицательная длительность, нулевая/чрезмерная длина, `played_ratio_pct` вне 0–100%;
- **duplicate** — полное повторение семи полей события;
- **review** — статистический или последовательностный кандидат, который нельзя удалять без бизнес-проверки: длинный трек, сверхчастый пользовательский бин, противоположные действия в одном бине, первое наблюдаемое событие `unlike`/`undislike`.

Такой подход не смешивает корректные структурные `null` и heavy-tail с настоящими ошибками качества.

In [ ]:
core_missing_expr = pl.any_horizontal(
    pl.col("uid").is_null(),
    pl.col("item_id").is_null(),
    pl.col("timestamp").is_null(),
    pl.col("is_organic").is_null(),
    pl.col("event_type").is_null(),
)
listen_expr = pl.col("event_type") == "listen"
structural_playback_expr = (
    (listen_expr & (pl.col("played_ratio_pct").is_null() | pl.col("track_length_seconds").is_null()))
    | (~listen_expr & (pl.col("played_ratio_pct").is_not_null() | pl.col("track_length_seconds").is_not_null()))
)

quality = events_lf.select(
    pl.len().alias("rows"),
    core_missing_expr.sum().alias("missing_core_fields"),
    (~pl.col("is_organic").is_in([0, 1])).sum().alias("invalid_is_organic"),
    (~pl.col("event_type").cast(pl.String).is_in(EVENT_TYPES)).sum().alias("invalid_event_type"),
    structural_playback_expr.sum().alias("structural_playback_mismatch"),
    (pl.col("played_ratio_pct").cast(pl.Int64) < 0).sum().alias("negative_played_ratio"),
    (pl.col("track_length_seconds").cast(pl.Int64) < 0).sum().alias("negative_track_length"),
    (listen_expr & (pl.col("played_ratio_pct") > 100)).sum().alias("played_ratio_over_100"),
    (listen_expr & (pl.col("track_length_seconds") <= 0)).sum().alias("nonpositive_track_length"),
    (listen_expr & (pl.col("track_length_seconds") > HARD_MAX_TRACK_SECONDS)).sum().alias("track_length_over_4h"),
    (listen_expr & (
        pl.col("track_length_seconds").cast(pl.Float64) * pl.col("played_ratio_pct") / 100
        > pl.col("track_length_seconds")
    )).sum().alias("played_seconds_over_track_length"),
    pl.struct(EXPECTED_COLUMNS).n_unique().alias("unique_rows"),
).collect(engine="streaming").to_dicts()[0]
quality["exact_duplicate_rows"] = quality["rows"] - quality["unique_rows"]

length_consistency_lf = (
    events_lf.filter(listen_expr)
    .group_by("item_id")
    .agg(pl.col("track_length_seconds").n_unique().alias("n_lengths"))
    .filter(pl.col("n_lengths") > 1)
)
items_with_inconsistent_length = length_consistency_lf.select(pl.len()).collect(engine="streaming").item()
length_p99 = int(listen_profile[0, "length_p99"])
long_tracks_above_p99 = (
    events_lf.filter(listen_expr & (pl.col("track_length_seconds") > length_p99))
    .select(pl.len()).collect(engine="streaming").item()
)
display(pl.DataFrame([quality]))
print(f"items with inconsistent length={items_with_inconsistent_length}; listens above length p99 ({length_p99}s)={long_tracks_above_p99:,}")

In [ ]:
actions_lf = events_lf.filter(pl.col("event_type") != "listen")
contradictory_bins_lf = (
    actions_lf.group_by(["uid", "item_id", "timestamp"])
    .agg(
        (pl.col("event_type") == "like").any().alias("has_like"),
        (pl.col("event_type") == "unlike").any().alias("has_unlike"),
        (pl.col("event_type") == "dislike").any().alias("has_dislike"),
        (pl.col("event_type") == "undislike").any().alias("has_undislike"),
    )
    .filter(
        (pl.col("has_like") & pl.col("has_unlike"))
        | (pl.col("has_dislike") & pl.col("has_undislike"))
    )
)
contradictory_bins = contradictory_bins_lf.select(pl.len()).collect(engine="streaming").item()

def state_family_profile(positive: str, reversal: str) -> dict:
    family = (
        actions_lf.filter(pl.col("event_type").is_in([positive, reversal]))
        .sort(["uid", "item_id", "timestamp", "event_type"])
    )
    firsts = family.group_by(["uid", "item_id"], maintain_order=True).agg(
        pl.col("event_type").first().alias("first_event")
    )
    repeated = family.with_columns(
        pl.col("event_type").shift(1).over(["uid", "item_id"]).alias("previous_event")
    ).filter(pl.col("event_type") == pl.col("previous_event"))
    return {
        "family": f"{positive}/{reversal}",
        "first_is_reversal": firsts.filter(pl.col("first_event") == reversal).select(pl.len()).collect(engine="streaming").item(),
        "repeated_same_transition": repeated.select(pl.len()).collect(engine="streaming").item(),
    }

like_state = state_family_profile("like", "unlike")
dislike_state = state_family_profile("dislike", "undislike")

burst_lf = events_lf.group_by(["uid", "timestamp"]).agg(
    pl.len().alias("events"),
    pl.col("item_id").n_unique().alias("items"),
    pl.col("event_type").n_unique().alias("event_types"),
)
burst_profile = burst_lf.select(
    pl.col("events").quantile(0.999, interpolation="nearest").alias("p999"),
    pl.col("events").max().alias("max"),
    (pl.col("events") > 20).sum().alias("bins_over_20"),
    (pl.col("events") > 100).sum().alias("bins_over_100"),
).collect(engine="streaming").to_dicts()[0]
top_bursts = burst_lf.sort("events", descending=True).limit(10).collect(engine="streaming")

print(like_state)
print(dislike_state)
print({"contradictory_bins": contradictory_bins, **burst_profile})
display(top_bursts)

In [ ]:
anomaly_summary = pl.DataFrame([
    {"category": "schema", "check": "missing_core_fields", "severity": "invalid", "threshold": "0", "candidates": quality["missing_core_fields"], "meaning": "Пропуск обязательного поля"},
    {"category": "schema", "check": "invalid_is_organic", "severity": "invalid", "threshold": "not in {0, 1}", "candidates": quality["invalid_is_organic"], "meaning": "Некорректный источник события"},
    {"category": "schema", "check": "invalid_event_type", "severity": "invalid", "threshold": "unknown enum", "candidates": quality["invalid_event_type"], "meaning": "Неизвестный тип события"},
    {"category": "schema", "check": "structural_playback_mismatch", "severity": "invalid", "threshold": "0", "candidates": quality["structural_playback_mismatch"], "meaning": "Playback-поля не соответствуют event_type"},
    {"category": "playback", "check": "negative_played_ratio", "severity": "invalid", "threshold": "< 0%", "candidates": quality["negative_played_ratio"], "meaning": "Отрицательная доля прослушивания"},
    {"category": "playback", "check": "negative_track_length", "severity": "invalid", "threshold": "< 0 sec", "candidates": quality["negative_track_length"], "meaning": "Отрицательная длина трека"},
    {"category": "playback", "check": "nonpositive_track_length", "severity": "invalid", "threshold": "<= 0 sec", "candidates": quality["nonpositive_track_length"], "meaning": "Нулевая или отрицательная длина"},
    {"category": "playback", "check": "track_length_over_4h", "severity": "invalid", "threshold": "> 14400 sec", "candidates": quality["track_length_over_4h"], "meaning": "Жёсткая верхняя граница длины"},
    {"category": "playback", "check": "played_ratio_over_100", "severity": "review", "threshold": "> 100%", "candidates": quality["played_ratio_over_100"], "meaning": "Расчётное время больше длины трека"},
    {"category": "playback", "check": "played_seconds_over_track_length", "severity": "review", "threshold": "> track length", "candidates": quality["played_seconds_over_track_length"], "meaning": "Время прослушивания вышло за физическую длину"},
    {"category": "playback", "check": "track_length_over_p99", "severity": "review", "threshold": f"> {length_p99} sec", "candidates": long_tracks_above_p99, "meaning": "Длинный контент; возможно не музыка"},
    {"category": "metadata", "check": "inconsistent_item_length", "severity": "invalid", "threshold": "> 1 length/item", "candidates": items_with_inconsistent_length, "meaning": "Один item имеет разные длины"},
    {"category": "duplicate", "check": "exact_duplicate_rows", "severity": "duplicate", "threshold": "full row", "candidates": quality["exact_duplicate_rows"], "meaning": "Повтор всех семи полей"},
    {"category": "sequence", "check": "contradictory_state_bin", "severity": "review", "threshold": "same 5-sec bin", "candidates": contradictory_bins, "meaning": "Like/unlike или dislike/undislike одновременно"},
    {"category": "sequence", "check": "first_event_is_unlike", "severity": "review", "threshold": "first observed", "candidates": like_state["first_is_reversal"], "meaning": "Может отражать состояние до окна"},
    {"category": "sequence", "check": "first_event_is_undislike", "severity": "review", "threshold": "first observed", "candidates": dislike_state["first_is_reversal"], "meaning": "Может отражать состояние до окна"},
    {"category": "sequence", "check": "repeated_like_state", "severity": "review", "threshold": "same transition twice", "candidates": like_state["repeated_same_transition"], "meaning": "Повтор like или unlike без смены состояния"},
    {"category": "sequence", "check": "repeated_dislike_state", "severity": "review", "threshold": "same transition twice", "candidates": dislike_state["repeated_same_transition"], "meaning": "Повтор dislike или undislike без смены состояния"},
    {"category": "velocity", "check": "user_bin_over_p999", "severity": "review", "threshold": "> 20 events/5 sec", "candidates": burst_profile["bins_over_20"], "meaning": "Бот, batch-загрузка или сбой timestamp"},
    {"category": "velocity", "check": "user_bin_over_100", "severity": "review", "threshold": "> 100 events/5 sec", "candidates": burst_profile["bins_over_100"], "meaning": "Экстремально невозможная ручная активность"},
]).sort("candidates", descending=True)
display(anomaly_summary)

plot_anomalies = anomaly_summary.filter(pl.col("candidates") > 0).sort("candidates")
colors = ["#F28E2B" if value == "invalid" else "#55C2D9" if value == "review" else "#B07AA1" for value in plot_anomalies["severity"]]
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh(plot_anomalies["check"], plot_anomalies["candidates"], color=colors)
axes[0].set_xscale("log")
axes[0].set_title("Кандидаты по проверкам (log)")
axes[0].set_xlabel("наблюдения / группы")
axes[0].grid(axis="x", alpha=0.25)

axes[1].barh([f"uid={u}" for u in top_bursts["uid"].cast(pl.String)], top_bursts["events"], color="#FFCC00")
axes[1].invert_yaxis()
axes[1].set_title("Топ пользовательских всплесков за 5 секунд")
axes[1].set_xlabel("события")
axes[1].grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "multi-event-anomalies.png", dpi=160, bbox_inches="tight")
plt.show()

## 5. Сохранение обогащённого датасета

Сырые значения не перезаписываются. Удаляются только точные полные дубликаты; к оставшимся строкам добавляются вычисляемые секунды и флаги. `played_ratio_pct > 100`, длинные треки, state-последовательности и всплески остаются доступными для расследования. Для безопасных агрегатов используется `played_seconds_capped`, а исходная оценка хранится в `played_seconds_raw`.

In [ ]:
ENRICHED_PATH = PROCESSED_DIR / "multi_event_enriched.parquet"
ROW_ANOMALIES_PATH = PROCESSED_DIR / "multi_event_row_anomalies.parquet"
ANOMALY_SUMMARY_PATH = PROCESSED_DIR / "multi_event_anomaly_summary.parquet"

enriched_lf = (
    analysis_lf.with_columns(
        core_missing_expr.alias("anomaly_missing_core"),
        structural_playback_expr.alias("anomaly_playback_structure"),
        (~pl.col("is_organic").is_in([0, 1])).alias("anomaly_invalid_is_organic"),
        (~pl.col("event_type").cast(pl.String).is_in(EVENT_TYPES)).alias("anomaly_invalid_event_type"),
        (listen_expr & (pl.col("played_ratio_pct").cast(pl.Int64) < 0)).alias("anomaly_negative_ratio"),
        (listen_expr & (pl.col("track_length_seconds").cast(pl.Int64) < 0)).alias("anomaly_negative_length"),
        (listen_expr & (pl.col("track_length_seconds") <= 0)).alias("anomaly_nonpositive_length"),
        (listen_expr & (pl.col("track_length_seconds") > HARD_MAX_TRACK_SECONDS)).alias("anomaly_hard_length"),
        (listen_expr & (pl.col("played_ratio_pct") > 100)).alias("anomaly_ratio_over_100"),
        (listen_expr & (pl.col("track_length_seconds") > length_p99)).alias("review_long_track"),
    )
    .unique(subset=EXPECTED_COLUMNS, keep="first", maintain_order=False)
)
enriched_lf.sink_parquet(ENRICHED_PATH, compression="zstd", mkdir=True)
(
    enriched_lf.filter(
        pl.any_horizontal(
            pl.col("anomaly_missing_core"),
            pl.col("anomaly_playback_structure"),
            pl.col("anomaly_invalid_is_organic"),
            pl.col("anomaly_invalid_event_type"),
            pl.col("anomaly_negative_ratio"),
            pl.col("anomaly_negative_length"),
            pl.col("anomaly_nonpositive_length"),
            pl.col("anomaly_hard_length"),
            pl.col("anomaly_ratio_over_100"),
            pl.col("review_long_track"),
        )
    )
    .sink_parquet(ROW_ANOMALIES_PATH, compression="zstd", mkdir=True)
)
anomaly_summary.write_parquet(ANOMALY_SUMMARY_PATH, compression="zstd")

saved_profile = pl.scan_parquet(ENRICHED_PATH).select(
    pl.len().alias("rows"),
    pl.col("anomaly_ratio_over_100").sum().alias("ratio_over_100"),
    pl.col("review_long_track").sum().alias("long_track_rows"),
).collect(engine="streaming")
display(saved_profile)
for path in [ENRICHED_PATH, ROW_ANOMALIES_PATH, ANOMALY_SUMMARY_PATH]:
    print(f"{path.relative_to(PROJECT_DIR)}: {path.stat().st_size / 1024**2:.2f} MiB")

## 6. Подготовка чистого датасета

Очистка следует доменной природе аномалий. Полные дубли и строки категории `invalid` удаляются. Прослушивания свыше 100% и длинный контент остаются с флагами и двумя оценками времени: `played_seconds_raw` для вовлечённости и `played_seconds_capped` для completion-метрик. Бины пользователя с более чем 100 событиями за 5 секунд исключаются из clean-слоя; бины с 21–100 событиями сохраняются как offline/batch sync, но не допускаются в sequence-модели.

Итоговый `multi_event_clean.parquet` не содержит точных дублей, invalid-строк и bot-бинов. Для аудита сохраняется таблица всех скоростных бинов, поэтому решение можно пересчитать с другими порогами без изменения raw-датасета.

In [ ]:
BOT_BIN_THRESHOLD = 100
OFFLINE_SYNC_THRESHOLD = 20
CLEAN_PATH = PROCESSED_DIR / "multi_event_clean.parquet"
VELOCITY_BINS_PATH = PROCESSED_DIR / "multi_event_velocity_bins.parquet"
CLEANING_SUMMARY_PATH = PROCESSED_DIR / "multi_event_cleaning_summary.parquet"

invalid_row_expr = pl.any_horizontal(
    core_missing_expr,
    ~pl.col("is_organic").is_in([0, 1]),
    ~pl.col("event_type").cast(pl.String).is_in(EVENT_TYPES),
    structural_playback_expr,
    listen_expr & (pl.col("played_ratio_pct").cast(pl.Int64) < 0),
    listen_expr & (pl.col("track_length_seconds").cast(pl.Int64) < 0),
    listen_expr & (pl.col("track_length_seconds") <= 0),
    listen_expr & (pl.col("track_length_seconds") > HARD_MAX_TRACK_SECONDS),
).fill_null(True)

# Сначала удаляем только полные технические дубли, затем invalid-строки.
valid_deduplicated_lf = (
    events_lf.unique(subset=EXPECTED_COLUMNS, keep="first", maintain_order=False)
    .filter(~invalid_row_expr)
)
velocity_flags_lf = (
    valid_deduplicated_lf.group_by(["uid", "timestamp"])
    .agg(
        pl.len().alias("events_in_5s"),
        pl.col("item_id").n_unique().alias("unique_items_in_5s"),
        pl.col("event_type").n_unique().alias("event_types_in_5s"),
    )
    .with_columns(
        (pl.col("events_in_5s") > BOT_BIN_THRESHOLD).alias("is_bot_session"),
        pl.col("events_in_5s").is_between(
            OFFLINE_SYNC_THRESHOLD + 1, BOT_BIN_THRESHOLD, closed="both"
        ).alias("is_offline_sync"),
    )
)
velocity_flags_lf.sink_parquet(VELOCITY_BINS_PATH, compression="zstd", mkdir=True)

suspected_bot_users_lf = (
    velocity_flags_lf.filter(pl.col("is_bot_session"))
    .select("uid").unique()
    .with_columns(pl.lit(True).alias("is_suspected_bot_user"))
)
curated_lf = (
    valid_deduplicated_lf
    .join(velocity_flags_lf, on=["uid", "timestamp"], how="left")
    .join(suspected_bot_users_lf, on="uid", how="left")
    .with_columns(
        pl.col("is_suspected_bot_user").fill_null(False),
        (pl.col("timestamp").cast(pl.UInt64) // 86400)
        .cast(pl.UInt32).alias("time_period"),
        (pl.col("event_type") == "listen").alias("is_listen"),
        (listen_expr & (pl.col("played_ratio_pct") > 100)).alias("is_replayed_over_100pct"),
        (listen_expr & (pl.col("track_length_seconds") > length_p99)).alias("is_long_content"),
        pl.when(listen_expr)
        .then(pl.col("track_length_seconds").cast(pl.Float64) * pl.col("played_ratio_pct") / 100)
        .otherwise(None).alias("played_seconds_raw"),
        pl.when(listen_expr)
        .then(pl.col("track_length_seconds").cast(pl.Float64) * pl.col("played_ratio_pct").clip(0, 100) / 100)
        .otherwise(None).alias("played_seconds_capped"),
    )
    .with_columns((~pl.col("is_bot_session") & ~pl.col("is_offline_sync")).alias("sequence_eligible"))
)

# Bot-бин исключается, но остальные события подозрительного uid сохраняются с отдельным флагом.
clean_lf = curated_lf.filter(~pl.col("is_bot_session"))
clean_lf.sink_parquet(CLEAN_PATH, compression="zstd", mkdir=True)

clean_profile = pl.scan_parquet(CLEAN_PATH).select(
    pl.len().alias("clean_rows"),
    pl.col("is_offline_sync").sum().alias("offline_sync_rows"),
    pl.col("is_suspected_bot_user").sum().alias("rows_of_suspected_bot_users"),
    pl.col("is_long_content").sum().alias("long_content_rows"),
    pl.col("is_replayed_over_100pct").sum().alias("replayed_over_100pct_rows"),
).collect(engine="streaming")
velocity_profile = pl.scan_parquet(VELOCITY_BINS_PATH).select(
    pl.col("is_bot_session").sum().alias("excluded_bot_bins"),
    pl.col("is_offline_sync").sum().alias("offline_sync_bins"),
).collect(engine="streaming")
display(clean_profile)
display(velocity_profile)

## 7. Нормализация состояний и Feature Store

Для `like/unlike` и `dislike/undislike` строится идемпотентный журнал переходов. Противоположные действия одного семейства в одном 5-секундном бине нейтрализуются. Если первой наблюдается отмена, добавляется синтетическое исходное положительное состояние с `state_timestamp = -1`. Повтор того же состояния без реальной смены игнорируется.

`state_snapshot_daily_sparse.parquet` — разреженный end-of-day snapshot: строка появляется только в день изменения состояния и действует до следующей строки той же пары. Это предотвращает взрыв объёма при полном декартовом заполнении всех дней. `state_snapshot_latest.parquet` хранит актуальные `is_liked` и `is_disliked` на конец окна наблюдения.

In [ ]:
STATE_TRANSITIONS_PATH = PROCESSED_DIR / "state_transitions_clean.parquet"
STATE_DAILY_PATH = PROCESSED_DIR / "state_snapshot_daily_sparse.parquet"
STATE_LATEST_PATH = PROCESSED_DIR / "state_snapshot_latest.parquet"
state_keys = ["uid", "item_id", "state_family"]

state_bins_lf = (
    pl.scan_parquet(CLEAN_PATH)
    .filter(pl.col("event_type") != "listen")
    .with_columns(
        pl.when(pl.col("event_type").cast(pl.String).is_in(["like", "unlike"]))
        .then(pl.lit("like")).otherwise(pl.lit("dislike")).alias("state_family"),
        pl.col("event_type").cast(pl.String).is_in(["like", "dislike"]).alias("state_value"),
    )
    .group_by(["uid", "item_id", "timestamp", "time_period", "state_family"])
    .agg(
        pl.col("state_value").n_unique().alias("states_in_bin"),
        pl.col("state_value").first().alias("state_value"),
        pl.col("is_offline_sync").any().alias("is_offline_sync"),
    )
    .filter(pl.col("states_in_bin") == 1)
    .drop("states_in_bin")
)
real_transitions_lf = state_bins_lf.select(
    *state_keys,
    pl.col("timestamp").cast(pl.Int64).alias("state_timestamp"),
    pl.col("time_period").cast(pl.Int64).alias("valid_from_day"),
    "state_value",
    pl.lit(False).alias("is_synthetic"),
    "is_offline_sync",
)
first_observed_lf = (
    state_bins_lf.sort([*state_keys, "timestamp"])
    .group_by(state_keys, maintain_order=True)
    .agg(pl.col("state_value").first().alias("first_state"))
)
implicit_initial_lf = first_observed_lf.filter(~pl.col("first_state")).select(
    *state_keys,
    pl.lit(-1, dtype=pl.Int64).alias("state_timestamp"),
    pl.lit(-1, dtype=pl.Int64).alias("valid_from_day"),
    pl.lit(True).alias("state_value"),
    pl.lit(True).alias("is_synthetic"),
    pl.lit(False).alias("is_offline_sync"),
)
state_transitions_lf = (
    pl.concat([real_transitions_lf, implicit_initial_lf], how="vertical_relaxed")
    .sort([*state_keys, "state_timestamp"])
    .with_columns(pl.col("state_value").shift(1).over(state_keys).alias("previous_state"))
    .filter(pl.col("previous_state").is_null() | (pl.col("state_value") != pl.col("previous_state")))
    .drop("previous_state")
    .with_columns(
        pl.when(pl.col("state_family") == "like")
        .then(pl.when(pl.col("state_value")).then(pl.lit("like")).otherwise(pl.lit("unlike")))
        .otherwise(pl.when(pl.col("state_value")).then(pl.lit("dislike")).otherwise(pl.lit("undislike")))
        .alias("normalized_event_type"),
        (~pl.col("is_offline_sync") & ~pl.col("is_synthetic")).alias("sequence_eligible"),
    )
)
state_transitions_lf.sink_parquet(STATE_TRANSITIONS_PATH, compression="zstd", mkdir=True)

In [ ]:
state_scan_lf = pl.scan_parquet(STATE_TRANSITIONS_PATH)
daily_state_lf = (
    state_scan_lf.group_by([*state_keys, "valid_from_day"])
    .agg(
        pl.col("state_value").sort_by("state_timestamp").last().alias("state_at_end_of_day"),
        pl.col("state_timestamp").max().alias("last_state_timestamp"),
        pl.col("is_synthetic").any().alias("contains_synthetic_transition"),
        pl.col("is_offline_sync").any().alias("contains_offline_sync_transition"),
    )
    .sort([*state_keys, "valid_from_day"])
)
daily_state_lf.sink_parquet(STATE_DAILY_PATH, compression="zstd", mkdir=True)

latest_state_lf = (
    state_scan_lf.group_by(["uid", "item_id"])
    .agg(
        pl.col("state_value").filter(pl.col("state_family") == "like")
        .sort_by(pl.col("state_timestamp").filter(pl.col("state_family") == "like")).last().alias("is_liked"),
        pl.col("state_value").filter(pl.col("state_family") == "dislike")
        .sort_by(pl.col("state_timestamp").filter(pl.col("state_family") == "dislike")).last().alias("is_disliked"),
        (pl.col("state_family") == "like").any().alias("like_state_observed"),
        (pl.col("state_family") == "dislike").any().alias("dislike_state_observed"),
        pl.col("state_timestamp").max().alias("last_state_timestamp"),
    )
    .with_columns(pl.col("is_liked").fill_null(False), pl.col("is_disliked").fill_null(False))
)
latest_state_lf.sink_parquet(STATE_LATEST_PATH, compression="zstd", mkdir=True)

state_profile = pl.scan_parquet(STATE_TRANSITIONS_PATH).select(
    pl.len().alias("normalized_transitions"),
    pl.col("is_synthetic").sum().alias("synthetic_initial_states"),
    pl.col("is_offline_sync").sum().alias("offline_sync_transitions"),
).collect(engine="streaming")
snapshot_profile = pl.DataFrame({
    "daily_sparse_rows": [pl.scan_parquet(STATE_DAILY_PATH).select(pl.len()).collect(engine="streaming").item()],
    "latest_snapshot_rows": [pl.scan_parquet(STATE_LATEST_PATH).select(pl.len()).collect(engine="streaming").item()],
})
display(state_profile)
display(snapshot_profile)

In [ ]:
clean_scan_lf = pl.scan_parquet(CLEAN_PATH)
clean_validation = clean_scan_lf.select(
    pl.len().alias("rows"),
    (pl.len() - pl.struct(EXPECTED_COLUMNS).n_unique()).alias("exact_duplicates"),
    pl.col("is_bot_session").sum().alias("bot_rows_remaining"),
    (pl.col("played_seconds_capped") > pl.col("track_length_seconds")).sum().alias("capped_seconds_over_length"),
    (pl.col("played_seconds_raw") < 0).sum().alias("negative_played_seconds"),
).collect(engine="streaming")
max_clean_bin = (
    clean_scan_lf.group_by(["uid", "timestamp"]).agg(pl.len().alias("events"))
    .select(pl.col("events").max()).collect(engine="streaming").item()
)
transition_validation = (
    pl.scan_parquet(STATE_TRANSITIONS_PATH).sort([*state_keys, "state_timestamp"])
    .with_columns(pl.col("state_value").shift(1).over(state_keys).alias("previous_state"))
    .select(
        (pl.col("state_value") == pl.col("previous_state")).sum().alias("repeated_states"),
        pl.struct([*state_keys, "state_timestamp"]).is_duplicated().sum().alias("duplicate_transition_keys"),
    ).collect(engine="streaming")
)
summary = pl.DataFrame({
    "metric": ["raw_rows", "exact_duplicates_removed", "invalid_rows_removed", "bot_bins_excluded", "clean_rows", "max_events_in_clean_5s_bin"],
    "value": [quality["rows"], quality["exact_duplicate_rows"], sum(quality[name] for name in ["missing_core_fields", "invalid_is_organic", "invalid_event_type", "structural_playback_mismatch", "negative_played_ratio", "negative_track_length", "nonpositive_track_length", "track_length_over_4h"]), velocity_profile[0, "excluded_bot_bins"], clean_validation[0, "rows"], max_clean_bin],
})
summary.write_parquet(CLEANING_SUMMARY_PATH, compression="zstd")
display(clean_validation)
display(transition_validation)
display(summary)

assert clean_validation[0, "exact_duplicates"] == 0
assert clean_validation[0, "bot_rows_remaining"] == 0
assert clean_validation[0, "capped_seconds_over_length"] == 0
assert clean_validation[0, "negative_played_seconds"] == 0
assert max_clean_bin <= BOT_BIN_THRESHOLD
assert transition_validation[0, "repeated_states"] == 0
assert transition_validation[0, "duplicate_transition_keys"] == 0

for path in [CLEAN_PATH, VELOCITY_BINS_PATH, STATE_TRANSITIONS_PATH, STATE_DAILY_PATH, STATE_LATEST_PATH, CLEANING_SUMMARY_PATH]:
    print(f"{path.relative_to(PROJECT_DIR)}: {path.stat().st_size / 1024**2:.2f} MiB")

## 8. Основные выводы

- В датасете 47,79 млн событий, 10 тыс. пользователей и 934 тыс. треков; `listen` доминирует над state-событиями.
- Структурные `null` playback-полей у событий действий корректны. Обязательные поля заполнены, отрицательных длительностей и долей нет благодаря беззнаковой схеме.
- 218 145 прослушиваний имеют `played_ratio_pct > 100`; расчётное время у них превышает длину трека. Они помечены, а для суммарного времени предусмотрено capped-поле.
- Найдено 236 230 полных дубликатов. Они удалены только в производных файлах, исходный DVC-артефакт не меняется.
- После дедупликации осталось 231 bot-бин с более чем 100 событиями. Из clean-слоя исключены только 56 438 строк этих бинов; остальные события подозрительных UID сохранены с `is_suspected_bot_user`.
- 1 200 664 события из бинов 21–100 сохранены как `is_offline_sync`, но исключены из sequence-задач через `sequence_eligible = false`.
- Длинные треки выше p99 не удаляются: 436 257 clean-строк помечены `is_long_content`; это может быть подкаст, микс или другой корректный длинный контент.
- Clean-слой содержит 47 497 781 строк, не имеет дублей, bot-бинов и нарушений capped-времени. Нормализовано 1 411 055 state-переходов, создано 213 520 неявных исходных состояний и 1 104 073 актуальных user-item snapshot-записей.

In [ ]:
max_period = clean_lf.select(pl.col("time_period").max()).collect().item()
print(max_period)

In [ ]:
gaps = (
    events_lf.sort(["uid", "timestamp"])
    .with_columns(pl.col("timestamp").diff().over("uid").alias("gap"))
    .filter(pl.col("gap") > 0)
    .select(pl.col("gap").min().alias("min_gap"))
    .collect()
)
print(gaps)